### Импортируем библиотеки

In [1]:
import numpy as np
import torch
from torch.nn import functional as F
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score
from scipy.stats import uniform, randint
import xgboost as xgb

### Загрузим данные, будем работать как через каггл, так и через свои ноутбук
Для работы со своим ноутбуком можно поставить *kaggle = False*

In [2]:
kaggle = False
dir = ''
if kaggle:
    dir = '/kaggle/input/ml-training/'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
data_train = pd.read_csv(dir + 'data/train.csv')
data_test = pd.read_csv(dir + 'data/test.csv')
data_train.shape, data_test.shape

((1288, 17), (430, 16))

### Просмотрим, что хранится в наших трейнах и тестах

In [4]:
data_train.head()

,area,perimeter,major_axis,minor_axis,eccentricity,eqdiasq,solidity,convex_area,extent,aspect_ratio,roundness,compactness,shapefactor_1,shapefactor_2,shapefactor_3,shapefactor_4,target
0,75516,1731.4840,411.7352,245.7620,0.8023,310.0806,0.9148,82546,0.7169,1.6753,0.3165,0.7531,0.0055,0.0033,0.5672,0.9502,1
1,98903,1374.4370,477.2451,269.7676,0.8249,354.8622,0.9585,103181,0.7679,1.7691,0.6579,0.7436,0.0048,0.0027,0.5529,0.9781,0
2,84746,1311.1570,482.7735,235.9040,0.8725,328.4843,0.9121,92914,0.7162,2.0465,0.6195,0.6804,0.0057,0.0028,0.4630,0.9474,1
3,98184,1463.1680,434.3769,292.6472,0.7390,353.5700,0.9543,102890,0.7316,1.4843,0.5763,0.8140,0.0044,0.0030,0.6625,0.9834,0
4,94170,1267.7271,440.1109,278.4162,0.7745,346.2672,0.9643,97656,0.6836,1.5808,0.7363,0.7868,0.0047,0.0030,0.6190,0.9785,0


In [5]:
data_test.head()

,area,perimeter,major_axis,minor_axis,eccentricity,eqdiasq,solidity,convex_area,extent,aspect_ratio,roundness,compactness,shapefactor_1,shapefactor_2,shapefactor_3,shapefactor_4
0,93313,1862.7260,447.1666,278.4893,0.7824,344.6880,0.9361,99678,0.7607,1.6057,0.3380,0.7708,0.0048,0.0030,0.5942,0.9541
1,78778,2159.5969,439.1004,240.0113,0.8374,316.7069,0.9340,84345,0.7265,1.8295,0.2123,0.7213,0.0056,0.0030,0.5202,0.9517
2,74757,1661.6720,441.6910,225.3914,0.8600,308.5183,0.9359,79873,0.7255,1.9597,0.3402,0.6985,0.0059,0.0030,0.4879,0.9561
3,88074,2199.8889,460.8836,283.3717,0.7886,334.8721,0.8709,101133,0.6922,1.6264,0.2287,0.7266,0.0052,0.0032,0.5279,0.8586
4,79318,2589.4900,429.3032,279.5817,0.7589,317.7905,0.8666,91528,0.6634,1.5355,0.1486,0.7402,0.0054,0.0035,0.5480,0.8414


### Разделим на трейны и на валидацию

In [6]:
X = data_train.drop('target', axis=1) # Признаки (все столбцы кроме 'target')
y = data_train['target'] 

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

#### Создадим Dataloader

In [7]:
class Binary_Dataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.targets = torch.tensor(targets.values, dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, ind):
        return self.features[ind], self.targets[ind]
    

class Test_Binary_Dataset(Dataset):
    def __init__(self, features):
        self.features = torch.tensor(features.values, dtype=torch.float32)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, ind):
        return self.features[ind], self.targets[ind]


# Create dataset
train_dataset = Binary_Dataset(X_train, y_train)
val_dataset = Binary_Dataset(X_val, y_val)
test_dataset = Test_Binary_Dataset(data_test)

# batch_size
batch_size = 64

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


### Обучим модель на xgboost, до этого они были подобраны через перебор

In [8]:
best_params = {
    'colsample_bytree': 0.7802367410867509,
    'gamma': 4.573478717606073,
    'learning_rate': 0.20733232141006055,
    'max_depth': 3,
    'min_child_weight': 8,
    'n_estimators': 1364,
    'subsample': 0.5905314495579822,
    'objective': 'binary:logistic',
    'use_label_encoder': False,
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'n_jobs': -1,
    'random_state': 41
}

final_xgb_model = xgb.XGBClassifier(**best_params)

final_xgb_model.fit(X, y)

X_test_prepared_np = data_test.to_numpy()

y_test_pred = final_xgb_model.predict(X_test_prepared_np)

# Преобразуем в int на всякий случай
y_test_pred = y_test_pred.astype(int)

submission = pd.DataFrame({'target': y_test_pred})

submission.to_csv('answers.csv', index=False, header=False) 
print("Сохранено")
print("Распределение предсказанных классов:")
print(submission['target'].value_counts(normalize=True))
    

c:\Users\redmi\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [22:00:00] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Сохранено
Распределение предсказанных классов:
target
1    0.606977
0    0.393023
Name: proportion, dtype: float64
